In [55]:
import os
from dotenv import load_dotenv
import requests
import base64 
import json
from urllib.parse import quote
import time
import pandas as pd
import billboard
from billboard import ChartData
from requests.adapters import HTTPAdapter, Retry



load_dotenv()

client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
base_url = "https://api.spotify.com/v1"
redirect_uri = os.getenv("REDIRECT_URI")
scope = "playlist-read-public"
token_url = "https://accounts.spotify.com/api/token"

class SpotifyAPI:
    def __init__(self) -> object:
        self.client_id = client_id
        self.client_secret = client_secret
        self.base_url = base_url
        self.token_url = token_url
        
    def get_access_token(self, max_retries=3, initial_delay=1):
        """
        Get Spotify access token with retry logic for JSON decode errors.
        """
        for attempt in range(1, max_retries + 1):
            try:
                response = requests.post(
                    self.token_url,
                    data={"grant_type": "client_credentials"},
                    auth=(self.client_id, self.client_secret),
                    timeout=10
                )
                
                if response.status_code == 200:
                    try:
                        return response.json()["access_token"]
                    except (ValueError, KeyError) as e:
                        if attempt < max_retries:
                            delay = initial_delay * (2 ** (attempt - 1))
                            print(f"Error parsing access token response: {e}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                            time.sleep(delay)
                            continue
                        else:
                            print(f"Error parsing access token response after {max_retries} attempts: {e}")
                            raise
                else:
                    if attempt < max_retries:
                        delay = initial_delay * (2 ** (attempt - 1))
                        print(f"Failed to get access token: HTTP {response.status_code}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                        time.sleep(delay)
                        continue
                    else:
                        print(f"Failed to get access token after {max_retries} attempts: HTTP {response.status_code}")
                        raise Exception(f"Failed to get access token: HTTP {response.status_code}")
                        
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < max_retries:
                    delay = initial_delay * (2 ** (attempt - 1))
                    print(f"Network error getting access token: {e}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"Network error getting access token after {max_retries} attempts: {e}")
                    raise
        
        raise Exception("Failed to get access token after all retries")

    def make_request(self, endpoint, params=None, body=None, max_retries=3, initial_delay=1):
        """
        Make a request to Spotify API with retry logic for JSON decode errors and rate limits.
        """
        url = f"{self.base_url}/{endpoint}"
        if params:
            for param in params:
                url+=f"?{param}"
        
        for attempt in range(1, max_retries + 1):
            try:
                access_token = self.get_access_token()
                res = requests.get(
                    url,
                    headers={
                        "Authorization" : f"Bearer {access_token}"
                    },
                    timeout=30
                )

                # Handle rate limiting (429)
                if res.status_code == 429:
                    if attempt < max_retries:
                        # Check for Retry-After header
                        retry_after = res.headers.get('Retry-After')
                        if retry_after:
                            try:
                                delay = int(retry_after) + 1
                            except ValueError:
                                delay = initial_delay * (2 ** (attempt - 1))
                        else:
                            delay = initial_delay * (2 ** (attempt - 1))
                        delay = min(delay, 60)  # Cap at 60 seconds
                        print(f"Rate limited (429). Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                        time.sleep(delay)
                        continue
                    else:
                        print(f"Rate limited (429). Max retries ({max_retries}) exceeded.")
                        return None

                if res.status_code == 200:
                    try:
                        return res.json()
                    except (ValueError, requests.exceptions.JSONDecodeError) as e:
                        if attempt < max_retries:
                            delay = initial_delay * (2 ** (attempt - 1))
                            print(f"JSON decode error: {e}. Response text: {res.text[:100]}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                            time.sleep(delay)
                            continue
                        else:
                            print(f"JSON decode error after {max_retries} attempts: {e}. Response text: {res.text[:200]}")
                            return None
                else:
                    if attempt < max_retries and res.status_code >= 500:
                        # Retry on server errors
                        delay = initial_delay * (2 ** (attempt - 1))
                        print(f"Server error {res.status_code}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                        time.sleep(delay)
                        continue
                    else:
                        print(f"Failed to fetch: {res.status_code} - {res.reason}")
                        return None
                        
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < max_retries:
                    delay = initial_delay * (2 ** (attempt - 1))
                    print(f"Network error: {e}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"Network error after {max_retries} attempts: {e}")
                    return None
        
        return None


    def get_categories(self, country="US", limit=2):
        return self.make_request(f"browse/categories?country={country}&limit={limit}")

    def get_playlists(self, category_id, country="US", limit=5):
        return self.make_request(f"search?q=category:{category_id}&type=playlist&limit={limit}")

    def search_song(self, song, artist):
        query = f"track:{song} artist:{artist}"
        endpoint = f"search?q={quote(query)}&type=track&limit=1"
        res = self.make_request(endpoint=endpoint)
        if not res:
            return None

        tracks = res.get("tracks", [])
        if not tracks:
            return None
 
        if tracks['total'] == 0:
            return -1
        id = self.make_request(endpoint=endpoint)['tracks']['items'][0]
        # print(id)
        return id
    def search_artist(self, artist):
        query = f"artist:{artist}"
        endpoint = f"search?q={quote(query)}&type=artist&limit=1"
        id = self.make_request(endpoint=endpoint)
        
        # return self.make_request(f"tracks/{id}")
        return id   
    def get_artist(self, id):
        return self.make_request(f"artists/{id}")
    



# base_url_last_fm="https://ws.audioscrobbler.com/2.0"
# last_fm_api_key = os.getenv("LAST_FM_API_KEY")
# last_fm_shared_secret = os.getenv("LAST_FM_SHARED_SECRET")
base = f"https://musicbrainz.org/ws/2"

session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    raise_on_status=False,
)

session.mount("https://", HTTPAdapter(max_retries=retries))



#Music Brainz API
class MusicBrainzAPI():
    def __init__(self) -> object:
        self.base = base
        self.session = requests.Session()

        
    # def make_request(self, endpoint):
    #     url = f"{base_url_last_fm}/{endpoint}&api_key={last_fm_api_key}&format=json"
    #     res = requests.get(url)
    #     print(url)
    #     return res.json()
    
    def make_request(self, endpoint, retries=3):
        url = f"{self.base}/{endpoint}&fmt=json"
        print(url)

        for attempt in range(1, retries + 1):
            try:
                res = self.session.get(url, timeout=20)

                # Handle rate limiting
                if res.status_code == 503:
                    wait = 1 * attempt
                    print(f"Rate limit hit. Retrying in {wait}s...")
                    time.sleep(wait)
                    continue

                if res.status_code != 200:
                    print(f"HTTP {res.status_code}: {res.reason}")
                    return None

                try:
                    return res.json()
                except ValueError:
                    print("Error decoding JSON")
                    return None

            except (requests.exceptions.Timeout,
                    requests.exceptions.ConnectionError) as e:
                print(f"Network error: {e}. Attempt {attempt}/{retries}")
                time.sleep(1)
        
        print("Max retries exceeded.")
        return None
    
    

    def get_mbid(self, link):
        endpoint = f"url?resource={link}&inc=artist-rels"
        res = self.make_request(endpoint)

        if not res:
            return None

        relations = res.get("relations", [])
        if not relations:
            return None

        artist = relations[0].get("artist", {})
        return artist.get("id")
    
    
    def mb_get_artist_tag(self, id):
        endpoint = f"artist/{id}?inc=tags"
        res = self.make_request(endpoint)
        if not res:
            return None
        tags = res.get('tags', [])
        if not tags:
            return None
        top = max(tags, key=lambda tag: tag.get('count')).get('name', "")
        return top



class BillBoardChart:
    def __init__(self, date):
        self.data = billboard.ChartData('hot-100', date=date)




base_recco="https://api.reccobeats.com/v1"
headers = {
  'Accept': 'application/json'
}


class ReccoBeats:
    def __init__(self):
        self.base=base_recco
        self.headers=headers

    def _make_request_with_retry(self, url, max_retries=5, initial_delay=1):
        ## Make a request with exponential backoff retry logic for rate limiting.
 
        for attempt in range(1, max_retries + 1):
            try:
                res = requests.get(url, headers=self.headers, timeout=30)
                
                if res.status_code == 200:
                    return res
                
                if res.status_code == 429:
                    if attempt < max_retries:
                        delay = initial_delay * (2 ** (attempt - 1))
                        # Cap at 60 seconds
                        delay = min(delay, 60)
                        
                        retry_after = res.headers.get('Retry-After')
                        if retry_after:
                            try:
                                delay = int(retry_after) + 1  
                            except ValueError:
                                pass
                        
                        print(f"Rate limited (429). Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                        time.sleep(delay)
                        continue
                    else:
                        print(f"Rate limited (429). Max retries ({max_retries}) exceeded.")
                        return res
                
                # Dont retry if other errors
                print(f"Request failed with code {res.status_code}: {res.reason}")
                return res
                
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < max_retries:
                    delay = initial_delay * (2 ** (attempt - 1))
                    print(f"Network error: {e}. Retrying in {delay}s (attempt {attempt}/{max_retries})...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"Network error after {max_retries} attempts: {e}")
                    return None
        
        return None

    def get_recco_song_details(self, ids):
        ids_string=','.join(ids)
        url = f"{self.base}/track?ids={ids_string}"
        res = self._make_request_with_retry(url)
        
        if res and res.status_code == 200:
            try:
                return res.json()['content']
            except (KeyError, ValueError) as e:
                print(f"Error parsing response: {e}")
                return None
        else:
            return None

    def get_recco_audio_analysis(self, id):
        url = f"{self.base}/track/{id}/audio-features"
        res = self._make_request_with_retry(url)
        
        if res and res.status_code == 200:
            try:
                return res.json()
            except ValueError as e:
                print(f"Error parsing audio analysis response for {id}: {e}")
                return None
        else:
            if res:
                print(f"Audio analysis request failed for {id} with code {res.status_code}: {res.reason}")
            return None

    def get_recco_artist_details(self, id):
        url = f"{self.base}/artist/{id}"
        res = self._make_request_with_retry(url)
        if res and res.status_code == 200:
            return res.json()
        return None


In [ ]:
from datetime import date, timedelta
import re
import unicodedata
import time
import pandas as pd
import sqlalchemy as sa


sp = SpotifyAPI()
mb = MusicBrainzAPI()
rec = ReccoBeats()


def get_spotify_song_ids_and_artists(chart) -> tuple[list, list, list, list]:
    """
    Returns:
        - chart: chart entries with song_id and artist_id (primary artist) added
        - song_ids: list of unique song IDs
        - artists: list of unique artist dicts
        - song_artists: list of (song_id, artist_id) tuples for all song-artist relationships
    """
    artists = []
    song_ids = []
    song_artists = []
    
    total = len(chart)
    print(f"  Searching Spotify for {total} songs...")
    
    for idx, song in enumerate(chart, 1):
        if idx % 10 == 0 or idx == 1:
            print(f"  Processing song {idx}/{total}: {song.get('title', 'Unknown')} by {song.get('artist', 'Unknown')}")
        artist = normalize_artist_name(song["artist"])
        res = sp.search_song(song["title"], artist)
        # add spotify ID and primary artist reference to each entry
        # initialize as null in case not found
        song["song_id"] = None
        song["artist_id"] = None

        # if the song is found, add song and artist
        if res and res != -1:
            track_id = res.get("id")
            track_artists = res.get("artists", [])

            if track_id:
                song_ids.append(track_id)
                song["song_id"] = track_id

            # store primary artist for the chart entry
            if track_artists:
                primary_artist = track_artists[0]
                song["artist_id"] = primary_artist.get("id")

            # maintain a de‑duplicated list of all artists we encounter
            # and track all song-artist relationships
            existing_ids = {a["id"] for a in artists}
            for item in track_artists:
                artist_id = item.get("id")
                artist_url = item.get("external_urls", {}).get("spotify")
                artist_name = item.get("name")
                
                if artist_id:
                    # Add to artists list if not seen before
                    if artist_id not in existing_ids:
                        artists.append(
                            {
                                "name": artist_name,
                                "id": artist_id,
                                "url": artist_url,
                            }
                        )
                        existing_ids.add(artist_id)
                    
                    # Track song-artist relationship
                    if track_id:
                        song_artists.append({
                            "song_id": track_id,
                            "artist_id": artist_id
                        })
    
    return (chart, song_ids, artists, song_artists)


def normalize_artist_name(artist) -> str:
    #special case
    if artist == "JEONGYEON, JIHYO & CHAEYOUNG Of TWICE":
        artist = "TWICE"
    #only need to get one artist if there are multipl
    else:
        for c in ["Featuring", ",", "&", ":", "With"]:
            if c.lower() in artist.lower():
                artist = artist.split(c)[0]
    
    return artist
    

def get_tags_music_brainz(artists):
    mbids = []
    for artist in artists:
        print(artist)

        url = artist['url']
        mbid = mb.get_mbid(url)
        if mbid:
            artist['mbid'] = mbid
            tag = mb.mb_get_artist_tag(mbid)
            if tag:
                artist['tag'] = tag
        else:
            print(f"unable to get mbid for {url}")
        time.sleep(1)


def get_audio_details(song_ids):
    """
    Fetch audio features for songs from ReccoBeats API.
    Includes retry logic with exponential backoff for 429 rate limit errors.
    """
    if not song_ids:
        print("No song IDs provided, returning empty dataframe")
        return pd.DataFrame()
    
    total = len(song_ids)
    print(f"Fetching audio details for {total} songs...")
    
    BATCH_SIZE = 40
    all_results = []
    
    # Fetch song details in batches
    num_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
    for batch_num, i in enumerate(range(0, len(song_ids), BATCH_SIZE), 1):
        batch = song_ids[i : i + BATCH_SIZE]
        print(f"  Fetching batch {batch_num}/{num_batches} ({len(batch)} songs)...")
        results = rec.get_recco_song_details(batch)
        if results:
            # Extract song_id and artist_ids from each result
            for r in results:
                # Extract song_id from href (e.g., "https://api.reccobeats.com/v1/track/123" -> "123")
                song_id = None
                if 'href' in r:
                    song_id = r['href'].split('/')[-1]
                
                # Extract artist_ids from artists array
                artist_ids = []
                if 'artists' in r and isinstance(r['artists'], list):
                    artist_ids = [
                        artist['href'].split('/')[-1] 
                        for artist in r['artists'] 
                        if 'href' in artist
                    ]
                
                # Add song_id and artist_ids to the result
                r['song_id'] = song_id
                r['artist_ids'] = artist_ids
                all_results.append(r)
            
            print(f"  Retrieved {len(results)} song details")
        else:
            print(f"  No results for batch {batch_num}")
        
        # Rate limiting - increased sleep time between batches to avoid 429 errors
        if batch_num < num_batches:
            time.sleep(2)  # Increased from 1 to 2 seconds

    if not all_results:
        print("No audio details retrieved, returning empty dataframe")
        return pd.DataFrame()

    print(f"Fetching audio analysis for {len(all_results)} songs...")
    # Fetch audio analysis for each song with rate limiting
    for idx, item in enumerate(all_results, 1):
        if idx % 10 == 0:
            print(f"  Processing audio analysis {idx}/{len(all_results)}...")
        # Use the song_id we extracted, or fall back to 'id' field
        track_id = item.get('id')
        if track_id:
            res = rec.get_recco_audio_analysis(track_id)
            if res:
                item.update(res)
        
        # Rate limiting - sleep between individual requests to avoid 429 errors
        # Only sleep if not the last item
        if idx < len(all_results):
            time.sleep(0.5)  # 500ms delay between requests
    
    print(f"Completed fetching audio details for {len(all_results)} songs")

    audio_features = pd.DataFrame(all_results)
    if audio_features.empty:
        return pd.DataFrame()
    
    # song_id and artist_ids are already extracted and added above
    # Drop unnecessary columns
    dropped = ["ean", "availableCountries", "isrc", "upc", "href"]
    # Also drop 'id' if it exists (we're using song_id instead)
    # if "id" in audio_features.columns and "song_id" in audio_features.columns:
    #     dropped.append("id")
    audio_features.drop(labels=dropped, inplace=True, errors="ignore")
    return audio_features



def most_recent_friday(ref_date=None):
    # Use today's date if none is provided
    if ref_date is None:
        ref_date = date.today()
    
    days_since_friday = (ref_date.weekday() - 4) % 7
    return ref_date - timedelta(days=days_since_friday)


def create_chart_object(chart):
    rows = []
    for entry in chart.entries:
        rows.append({
            "title": entry.title,
            "artist": entry.artist,
            "rank": entry.rank,
            "isNew": entry.isNew,
            "weeks": entry.weeks,
            "peakPos": entry.peakPos
        })
    return rows


def get_dataframes(chart_week: str):
    """
    Build dataframes that align with the schema:
    - chart_weeks (PK: chart_week)
    - chart_entries (PK: chart_week, rank)
    - songs (PK: song_id)
    - artists (PK: artist_id)
    - song_artists (PK: song_id, artist_id) - many-to-many relationship
    """
    temp = BillBoardChart(chart_week).data
    chart = create_chart_object(temp)

    chart, song_ids, artists, song_artists = get_spotify_song_ids_and_artists(chart)
    get_tags_music_brainz(artists)

    df_audio = get_audio_details(song_ids)
    df_artists = pd.DataFrame(artists)
    df_chart = pd.DataFrame(chart)

    # add chart_week and normalize id column names
    df_chart["chart_week"] = chart_week

    if not df_artists.empty:
        df_artists.rename(columns={"id": "artist_id"}, inplace=True)

    # songs table
    # one row per song with FKs into audio features and artists (primary artist)
    if not df_chart.empty and "song_id" in df_chart.columns and "title" in df_chart.columns:
        songs_with_ids = df_chart[df_chart["song_id"].notna()][["song_id", "title"]]
        if not songs_with_ids.empty:
            if df_audio is not None and not df_audio.empty:
                songs = (
                    songs_with_ids
                    .merge(df_audio, on="song_id", how="left")
                    .drop_duplicates(subset=["song_id"])
                    .reset_index(drop=True)
                )
            else:
                # If no audio features, just use the song_id and title
                songs = (
                    songs_with_ids
                    .drop_duplicates(subset=["song_id"])
                    .reset_index(drop=True)
                )
        else:
            # No songs found, create empty dataframe
            songs = pd.DataFrame(columns=["song_id", "title"])
            if df_audio is not None and not df_audio.empty:
                for col in df_audio.columns:
                    if col != "song_id":
                        songs[col] = None
    else:
        # Chart is empty
        songs = pd.DataFrame(columns=["song_id", "title"])
        if df_audio is not None and not df_audio.empty:
            for col in df_audio.columns:
                if col != "song_id":
                    songs[col] = None

    # artists table
    # one row per artist with FKs into songs and artists (primary artist)
    if not df_artists.empty:
        artists_df = (
            df_artists.drop_duplicates(subset=["artist_id"]).reset_index(drop=True)
        )
    else:
        artists_df = pd.DataFrame(columns=["artist_id", "name", "url", "mbid", "tag"])

    # song_artists table 
    # one row per (song_id, artist_id) for many-to-many relationship
    if song_artists:
        song_artists_df = (
            pd.DataFrame(song_artists)
            .drop_duplicates(subset=["song_id", "artist_id"])
            .reset_index(drop=True)
        )
    else:
        song_artists_df = pd.DataFrame(columns=["song_id", "artist_id"])


    # chart_weeks table
    # one row per chart week
    chart_weeks = pd.DataFrame({"chart_week": [chart_week]})


    # chart_entries table
    # one row per (chart_week, rank) with FKs into songs and artists (primary artist)
    required_cols = ["chart_week", "rank", "song_id", "artist_id", "isNew", "weeks", "peakPos", "title", "artist"]
    for col in required_cols:
        if col not in df_chart.columns:
            df_chart[col] = None
    
    chart_entries = df_chart[required_cols].copy()

    return {
        "chart_weeks": chart_weeks,
        "chart_entries": chart_entries,
        "songs": songs,
        "artists": artists_df,
        "song_artists": song_artists_df,
    }


Pandas version: 2.3.3
SQLAlchemy version: 1.4.54


In [60]:
date = "2025-11-28"
dfs = get_dataframes(date)
# temp = BillBoardChart(date).data

# chart = create_chart_object(temp)

# chart, song_ids, artists, song_artists = get_spotify_song_ids_and_artists(chart)
# # get_tags_music_brainz(artists)
# df_audio = get_audio_details(song_ids)
# df_chart = pd.DataFrame(chart)

#     # add chart_week and normalize id column names
# df_chart["chart_week"] = date

# if not df_chart.empty and "song_id" in df_chart.columns and "title" in df_chart.columns:
#     print("if")
#     songs_with_ids = df_chart[df_chart["song_id"].notna()][["song_id", "title"]]
#     if not songs_with_ids.empty:
#         if df_audio is not None and not df_audio.empty:
#             songs = (
#                 songs_with_ids
#                     .merge(df_audio, on="song_id", how="left")
#                     .drop_duplicates(subset=["song_id"])
#                     .reset_index(drop=True)
#                 )
#         else:
#                 # If no audio features, just use the song_id and title
#                 songs = (
#                     songs_with_ids
#                     .drop_duplicates(subset=["song_id"])
#                     .reset_index(drop=True)
#                 )
#     else:
#             # No songs found, create empty dataframe
#         songs = pd.DataFrame(columns=["song_id", "title"])
#         if df_audio is not None and not df_audio.empty:
#             for col in df_audio.columns:
#                 if col != "song_id":
#                     songs[col] = None
# else:
#     print("else")

#         # Chart is empty
#     songs = pd.DataFrame(columns=["song_id", "title"])
#     if df_audio is not None and not df_audio.empty:
#         for col in df_audio.columns:
#             if col != "song_id":
#                 songs[col] = None



  Searching Spotify for 100 songs...
  Processing song 1/100: The Fate Of Ophelia by Taylor Swift
  Processing song 10/100: I Got Better by Morgan Wallen
  Processing song 20/100: Manchild by Sabrina Carpenter
  Processing song 30/100: It's The Most Wonderful Time Of The Year by Andy Williams
  Processing song 40/100: A Holly Jolly Christmas by Burl Ives
  Processing song 50/100: 20 Cigarettes by Morgan Wallen
  Processing song 60/100: Go Girl by Summer Walker, Latto & Doja Cat
  Processing song 70/100: Sorry by NF With James Arthur
  Processing song 80/100: 1-800 Heartbreak by Summer Walker & Anderson .Paak
  Processing song 90/100: The Dead Dance by Lady Gaga
  Processing song 100/100: How Far Does A Goodbye Go by Jason Aldean
{'name': 'Taylor Swift', 'id': '06HL4z0CvFAxyc27GXpf02', 'url': 'https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02'}
https://musicbrainz.org/ws/2/url?resource=https://open.spotify.com/artist/06HL4z0CvFAxyc27GXpf02&inc=artist-rels&fmt=json
https://musicbrai

In [64]:
# songs_with_ids
# temp_songs = ( songs_with_ids
#                     .merge(df_audio, on="song_id",how ="right")
#                     # .drop_duplicates(subset=["song_id"])
#                     .reset_index(drop=True)
#                 )
                
# # temp_songs.head()
# # songs_with_ids
# # df_audio
# songs.head()
# print("hello")

songs = dfs['songs']
songs.head()

,song_id,title,id,trackTitle,artists,durationMs,isrc,ean,upc,href,...,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence
0,53iuhJlwXhSER5J2IYYv1W,The Fate Of Ophelia,81870dae-2a92-4e1e-9172-3e0a0494d6f1,The Fate of Ophelia,[{'id': 'c7b330b5-a62e-420c-bf02-943ca6bb8746'...,226073.0,USUG12506436,None,None,https://open.spotify.com/track/53iuhJlwXhSER5J...,...,0.726,0.598,0.000000,5.0,0.1020,-5.179,1.0,0.0291,123.921,0.519
1,1CPZ5BxNNd0n0nF4Orb9JS,Golden,4fd20ad6-564f-4304-9b22-3da9241798ac,Golden,[{'id': 'cde522a1-f5ff-4c30-b89f-1a64b95057f7'...,194607.0,QZ8BZ2513510,None,None,https://open.spotify.com/track/1CPZ5BxNNd0n0nF...,...,0.651,0.686,0.000000,4.0,0.1850,-4.371,0.0,0.0616,122.715,0.104
2,2RkZ5LkEzeHGRsmDqKwmaJ,Ordinary,43a9e618-d36c-424f-a9cc-5d6da569e257,Ordinary,[{'id': 'a0efa239-c432-4a69-ab56-e71de2e89278'...,186964.0,USAT22500463,None,None,https://open.spotify.com/track/2RkZ5LkEzeHGRsm...,...,0.368,0.694,0.000007,2.0,0.0550,-6.141,1.0,0.0600,168.115,0.391
3,1qbmS6ep2hbBRaEZFpn7BX,Man I Need,4128e2e9-00b2-4028-94a6-396ba44634b2,Man I Need,[{'id': '0a10f4b0-eab0-4548-abae-960b32cebb8d'...,184000.0,GBUM72503089,None,None,https://open.spotify.com/track/1qbmS6ep2hbBRaE...,...,0.808,0.593,0.000451,1.0,0.0933,-7.270,1.0,0.0714,119.008,0.688
4,3yWuTOYDztXjZxdE2cIRUa,Opalite,bc916d89-b433-4297-98b6-a18e9953b1e1,Opalite,[{'id': 'c7b330b5-a62e-420c-bf02-943ca6bb8746'...,235355.0,USUG12506438,None,None,https://open.spotify.com/track/3yWuTOYDztXjZxd...,...,0.806,0.819,0.000000,4.0,0.0371,-5.033,0.0,0.0384,124.994,0.802
